# Projekt: Analityka Diamentów - wycena diamentów
### Integracja, czyszczenie i przygotowanie danych + model ML (Przetwarzanie Danych)

**Źródło danych:** prawdziwy, publiczny zbiór **diamonds** (ggplot2/tidyverse, Hadley Wickham), 53 940 rekordów, pobrany bez żadnych zmian z:
https://raw.githubusercontent.com/tidyverse/ggplot2/main/data-raw/diamonds.csv

### Słowniczek pojęć (diamenty i ich parametry)

Zbiór opisuje diamenty przez tzw. **4C** (cztery kluczowe parametry jakości) oraz wymiary fizyczne. Poniższe pojęcia pojawiają się wielokrotnie w dalszej części notebooka:

| Pojęcie | Znaczenie |
|---|---|
| **carat** (waga) | Jednostka masy kamieni szlachetnych. **1 karat = 0,2 grama = 200 miligramów.** Główny czynnik determinujący cenę. |
| **cut** (szlif) | Jakość oszlifowania — jak precyzyjnie i proporcjonalnie oszlifowano kamień. Skala porządkowa: Fair < Good < Very Good < Premium < Ideal. |
| **color** (kolor) | Stopień bezbarwności. Skala porządkowa od D (całkowicie bezbarwny, najlepszy) do J (widoczny żółtawy odcień, najsłabszy w tym zbiorze). |
| **clarity** (czystość) | Ilość i widoczność wewnętrznych skaz (inkluzji). Skala porządkowa od I1 (skazy widoczne gołym okiem, najsłabsza) do IF ("internally flawless", brak skaz pod powiększeniem, najlepsza). |
| **depth** (głębokość) | Wysokość diamentu (od najniższego punktu do płaszczyzny table) wyrażona jako procent średniej średnicy kamienia. |
| **table** | Największa, płaska faseta na szczycie oszlifowanego diamentu — "okno", przez które widać wnętrze kamienia. Wartość w danych to procent: szerokość tej fasety względem całkowitej średnicy diamentu. |
| **x, y, z** | Wymiary fizyczne diamentu w milimetrach: długość, szerokość i wysokość. |

Praktyczna zależność wykorzystywana dalej do kontroli spójności danych: `depth ≈ 2 × z / (x + y) × 100` — głębokość da się w przybliżeniu wyliczyć z wymiarów fizycznych, więc rozbieżność między podaną a przeliczoną wartością sygnalizuje błąd pomiaru.

## Krok 0 - Importy i wczytanie danych (bez modyfikacji źródła)

In [1]:
import json
import os

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest, RandomForestRegressor
from sklearn.manifold import TSNE
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import LocalOutlierFactor
from sklearn.decomposition import PCA
from sklearn.preprocessing import RobustScaler

np.random.seed(42)
RAW_PATH = "diamonds.csv"
ART = "artifacts"
os.makedirs(ART, exist_ok=True)

## KROK 0 - Wczytanie danych (bez zadnych modyfikacji zrodla)

In [2]:
df = pd.read_csv(RAW_PATH)
n_rows_raw = len(df)

## Krok 1 - Usunięcie duplikatów

146 pełnych duplikatów wierszy, naturalnie obecnych w oryginalnym pliku.

In [3]:
n_duplicates = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)

## Krok 2 - Diagnoza (wykrycie, nie naprawa) anomalii pomiarowych

Zerowe wymiary x/y/z (fizycznie niemożliwe) oraz niespójność `depth` vs przeliczone z x/y/z (błąd transkrypcji). To tylko **wykrycie** — flagi logiczne bez żadnych parametrów estymowanych z danych, więc bezpieczne do wykonania przed podziałem train/test. Podejrzane wartości maskujemy na NaN, ale jeszcze ich NIE uzupełniamy — to nastąpi dopiero po podziale (Krok 4).

In [4]:
zero_dim_mask = (df["x"] == 0) | (df["y"] == 0) | (df["z"] == 0)
n_zero_dim = int(zero_dim_mask.sum())

depth_calc_raw = df["z"] / ((df["x"] + df["y"]) / 2) * 100
depth_mismatch_raw = (df["depth"] - depth_calc_raw).abs()
inconsistent_mask = (depth_mismatch_raw > 5) & (~zero_dim_mask)
n_inconsistent = int(inconsistent_mask.sum())

suspect_mask = zero_dim_mask | inconsistent_mask
for col in ["x", "y", "z"]:
    df.loc[suspect_mask, col] = np.nan

missing_cols = ["x", "y", "z"]
missing_before = df[missing_cols].isnull().sum().to_dict()

## Krok 3 - Podział Train/Test — PRZESUNIĘTY na wczesny etap

Podział 80/20 następuje TERAZ, zanim jakakolwiek transformacja "uczy się" z danych (mediana do imputacji, RobustScaler, PCA). Od tego miejsca każda taka transformacja będzie dopasowywana wyłącznie na indeksach treningowych.

In [5]:

train_idx, test_idx = train_test_split(df.index, test_size=0.2, random_state=42)
is_train = pd.Series(False, index=df.index)
is_train.loc[train_idx] = True

## Krok 4 - Imputacja braków — mediana liczona wyłącznie na zbiorze treningowym

Mediana w obrębie grupy `cut` obliczona TYLKO na rekordach treningowych, a następnie zastosowana do uzupełnienia braków zarówno w train, jak i w test. Test nigdy nie wpływa na wartość, którą się go "naprawia".

In [6]:
train_medians_by_cut = df.loc[is_train].groupby("cut")[["x", "y", "z"]].median()
global_train_median = df.loc[is_train, ["x", "y", "z"]].median()

for col in ["x", "y", "z"]:
    def fill_val(row):
        if pd.notnull(row[col]):
            return row[col]
        m = train_medians_by_cut.loc[row["cut"], col] if row["cut"] in train_medians_by_cut.index else np.nan
        return m if pd.notnull(m) else global_train_median[col]
    df[col] = df.apply(fill_val, axis=1)

missing_after = df[missing_cols].isnull().sum().to_dict()

## Krok 5 - Integracja danych — agregaty liczone wyłącznie na TRAIN

`clarity_summary` i `color_summary` (średnia cena wg grupy) budowane WYŁĄCZNIE z rekordów treningowych, dołączone metodą `merge` do całej tabeli. Te cechy (`delta_price_vs_*`) i tak są później wykluczone z modelu jako pochodne ceny — liczymy je poprawnie głównie dla higieny metodologicznej.

In [7]:

clarity_summary = df.loc[is_train].groupby("clarity").agg(clarity_avg_price=("price", "mean")).reset_index()
color_summary = df.loc[is_train].groupby("color").agg(color_avg_price=("price", "mean")).reset_index()
df = df.merge(clarity_summary, on="clarity", how="left")
df = df.merge(color_summary, on="color", how="left")

df["clarity_avg_price"] = df["clarity_avg_price"].fillna(df.loc[is_train, "price"].mean())
df["color_avg_price"] = df["color_avg_price"].fillna(df.loc[is_train, "price"].mean())
df["delta_price_vs_clarity"] = df["price"] - df["clarity_avg_price"]
df["delta_price_vs_color"] = df["price"] - df["color_avg_price"]

## Krok 6 - Detekcja anomalii cenowych (1): model bazowy + IQR

Model bazowy (cena ≈ a×carat^b) i progi IQR liczone na TRAIN, zastosowane do całego zbioru. To narzędzie opisowe (dashboard), a nie cecha modelu — jego wyjścia są jawnie wykluczone z cech ML, więc to nie jest wyciek danych do ewaluacji modelu, w odróżnieniu od imputacji/skalowania.

In [8]:
_log_carat_fit = np.log(df.loc[is_train, "carat"])
_log_price_fit = np.log(df.loc[is_train, "price"])
_b1, _b0 = np.polyfit(_log_carat_fit, _log_price_fit, deg=1)
df["price_baseline_pred"] = np.exp(_b0 + _b1 * np.log(df["carat"]))
df["price_residual"] = df["price"] - df["price_baseline_pred"]
df["price_residual_pct"] = df["price_residual"] / df["price_baseline_pred"]

Q1 = df.loc[is_train, "price_residual_pct"].quantile(0.25)
Q3 = df.loc[is_train, "price_residual_pct"].quantile(0.75)
IQR = Q3 - Q1
resid_upper = Q3 + 1.5 * IQR
resid_lower = Q1 - 1.5 * IQR
df["Is_Overpriced"] = df["price_residual_pct"] > resid_upper

## Krok 7 - Detekcja anomalii cenowych (2): Isolation Forest / LOF

In [9]:
df["Is_Undervalued"] = df["price_residual_pct"] < resid_lower

anomaly_features = df[["carat", "price", "price_residual_pct", "depth", "table"]].copy()
anomaly_features = (anomaly_features - anomaly_features.mean()) / anomaly_features.std()

iso = IsolationForest(contamination=0.05, random_state=42)
df["anomaly_isoforest"] = iso.fit_predict(anomaly_features) == -1

lof = LocalOutlierFactor(n_neighbors=20, contamination=0.05)
df["anomaly_lof"] = lof.fit_predict(anomaly_features) == -1

label = df["Is_Overpriced"] | df["Is_Undervalued"]

def precision_recall(pred, true):
    tp = (pred & true).sum()
    fp = (pred & ~true).sum()
    fn = (~pred & true).sum()
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    return precision, recall

p_iso, r_iso = precision_recall(df["anomaly_isoforest"], label)
p_lof, r_lof = precision_recall(df["anomaly_lof"], label)
anomaly_summary = pd.DataFrame(
    {"method": ["IsolationForest", "LOF"], "precision": [p_iso, p_lof], "recall": [r_iso, r_lof]}
)

## Krok 8 - Inżynieria cech

Wyłącznie deterministyczne formuły (volume, price_per_carat, depth_calculated, logi) — bez parametrów estymowanych z danych, bezpieczne niezależnie od kolejności względem podziału.

In [10]:
df["volume"] = df["x"] * df["y"] * df["z"]
df["price_per_carat"] = df["price"] / df["carat"]
df["depth_calculated"] = df["z"] / ((df["x"] + df["y"]) / 2) * 100
df["depth_mismatch"] = (df["depth"] - df["depth_calculated"]).abs()
df["log_price"] = np.log1p(df["price"])
df["log_carat"] = np.log1p(df["carat"])

## Krok 9 - Kodowanie zmiennych kategorycznych

Ordinal Encoding dla cut/color/clarity — mapy STAŁE, nie estymowane z danych.

In [11]:
cut_order = {"Fair": 0, "Good": 1, "Very Good": 2, "Premium": 3, "Ideal": 4}
color_order = {"J": 0, "I": 1, "H": 2, "G": 3, "F": 4, "E": 5, "D": 6}
clarity_order = {"I1": 0, "SI2": 1, "SI1": 2, "VS2": 3, "VS1": 4, "VVS2": 5, "VVS1": 6, "IF": 7}

df["cut_encoded"] = df["cut"].map(cut_order)
df["color_encoded"] = df["color"].map(color_order)
df["clarity_encoded"] = df["clarity"].map(clarity_order)

df_model_ready = df.drop(columns=["cut", "color", "clarity"])

## Krok 10 - Normalizacja — RobustScaler dopasowany WYŁĄCZNIE na train

**Druga kluczowa poprawka.** `scaler.fit()` wywołany tylko na `X_train`; `transform()` zastosowany do całego zbioru tymi samymi, ustalonymi na treningu parametrami (`center_`, `scale_`).

In [12]:
numeric_cols = [
    "carat", "depth", "table", "price", "x", "y", "z",
    "clarity_avg_price", "color_avg_price", "delta_price_vs_clarity", "delta_price_vs_color",
    "price_baseline_pred", "price_residual", "price_residual_pct",
    "volume", "price_per_carat", "depth_calculated", "depth_mismatch",
    "log_price", "log_carat", "cut_encoded", "color_encoded", "clarity_encoded",
]
scaler = RobustScaler()
scaler.fit(df_model_ready.loc[is_train, numeric_cols])  # <-- fit WYLACZNIE na train

df_scaled = df_model_ready.copy()
df_scaled[numeric_cols] = scaler.transform(df_model_ready[numeric_cols])  # transform na wszystkim

## Krok 11 - PCA — dopasowany wyłącznie na train

Główne składowe wyznaczone na `X_train`, a następnie zastosowane (`transform`) do próbki użytej w wizualizacji.

In [13]:
feature_cols_final = list(df_scaled.columns)

pca_full = PCA(random_state=42)
pca_full.fit(df_scaled.loc[is_train, feature_cols_final].values)  # <-- fit na train
cum_var = np.cumsum(pca_full.explained_variance_ratio_)
n_components_90 = int(np.argmax(cum_var >= 0.90) + 1)
pca_scree = pd.DataFrame({"n_components": np.arange(1, len(cum_var) + 1), "cumulative_variance": cum_var})

sample_idx = df_scaled.sample(n=min(6000, len(df_scaled)), random_state=42).index
pca_2 = PCA(n_components=2, random_state=42)
pca_2.fit(df_scaled.loc[is_train, feature_cols_final].values)  # <-- fit na train
pca_coords = pca_2.transform(df_scaled.loc[sample_idx, feature_cols_final].values)  # transform na probie do wizualizacji
pca_2d = pd.DataFrame(pca_coords, columns=["PC1", "PC2"])
pca_2d["cut"] = df.loc[sample_idx, "cut"].values
pca_2d["color"] = df.loc[sample_idx, "color"].values

## Krok 12 - t-SNE — czysto opisowa wizualizacja

t-SNE nie ma osobnej metody fit/transform na nowych danych — to wyłącznie narzędzie wizualizacyjne, nie karmi modelu, więc brak podziału train/test na tym etapie nie wpływa na ewaluację modelu ML.

In [14]:

tsne_idx = df_scaled.sample(n=min(2500, len(df_scaled)), random_state=42).index
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
tsne_coords = tsne.fit_transform(df_scaled.loc[tsne_idx, feature_cols_final].values)
tsne_2d = pd.DataFrame(tsne_coords, columns=["TSNE1", "TSNE2"])
tsne_2d["cut"] = df.loc[tsne_idx, "cut"].values
tsne_2d["color"] = df.loc[tsne_idx, "color"].values

## Krok 13 - Trening modelu

Podział train/test już gotowy z Kroku 3 — tu tylko budujemy macierze X/y i trenujemy `RandomForestRegressor` WYŁĄCZNIE na `X_train`/`y_train`.

In [15]:
leakage_cols = [
    "price", "price_per_carat", "delta_price_vs_clarity", "delta_price_vs_color",
    "price_baseline_pred", "price_residual", "price_residual_pct",
    "clarity_avg_price", "color_avg_price", "Is_Overpriced", "Is_Undervalued",
    "anomaly_isoforest", "anomaly_lof",
    "x", "y", "z"  # 
]
target = np.log1p(df["price"])
feature_cols_model = [c for c in feature_cols_final if c not in leakage_cols + ["log_price"]]

X = df_scaled[feature_cols_model]
y = target

X_train, X_test = X.loc[train_idx], X.loc[test_idx]
y_train, y_test = y.loc[train_idx], y.loc[test_idx]
idx_train, idx_test = train_idx, test_idx

model = RandomForestRegressor(n_estimators=120, max_depth=10, min_samples_leaf=3, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

RandomForestRegressor(max_depth=10, min_samples_leaf=3, n_estimators=120,
                      n_jobs=-1, random_state=42)

## Krok 14 - Ewaluacja modelu

R², MAE, RMSE liczone WYŁĄCZNIE na `X_test`/`y_test` — przeskalowanych parametrami wyznaczonymi na treningu, nigdy nie "widzianymi" przez żadną dopasowywaną transformację.

In [16]:
y_pred = model.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5

feature_importance = pd.DataFrame(
    {"feature": feature_cols_model, "importance": model.feature_importances_}
).sort_values("importance", ascending=False).reset_index(drop=True)

predictions = pd.DataFrame({"y_true": y_test.values, "y_pred": y_pred})
baseline_r2 = r2_score(df["price"], df["price_baseline_pred"])

## Krok 15 - Zapis artefaktów dla dashboardu

In [17]:
eda_overview = {
    "n_rows_raw": int(n_rows_raw),
    "n_duplicates_removed": int(n_duplicates),
    "n_rows": int(len(df)),
    "n_cols_final": int(len(feature_cols_model)),
    "n_zero_dimension_rows": n_zero_dim,
    "n_inconsistent_measurement_rows": n_inconsistent,
    "n_overpriced": int(df["Is_Overpriced"].sum()),
    "n_undervalued": int(df["Is_Undervalued"].sum()),
    "missing_after_masking": {k: int(v) for k, v in missing_before.items()},
    "missing_after_imputation": {k: int(v) for k, v in missing_after.items()},
    "pca_n_components_90pct": int(n_components_90),
    "baseline_r2_carat_only": float(baseline_r2),
    "corr_price_carat": float(df[["price", "carat"]].corr().iloc[0, 1]),
}
with open(os.path.join(ART, "eda_overview.json"), "w", encoding="utf-8") as f:
    json.dump(eda_overview, f, ensure_ascii=False, indent=2)
with open(os.path.join(ART, "model_metrics.json"), "w", encoding="utf-8") as f:
    json.dump({"r2": float(r2), "mae": float(mae), "rmse": float(rmse)}, f, indent=2)

df_sample = df.sample(n=min(8000, len(df)), random_state=42).reset_index(drop=True)
df_sample.to_csv(os.path.join(ART, "df_sample.csv"), index=False)
anomaly_summary.to_csv(os.path.join(ART, "anomaly_summary.csv"), index=False)
pca_scree.to_csv(os.path.join(ART, "pca_scree.csv"), index=False)
pca_2d.to_csv(os.path.join(ART, "pca_2d.csv"), index=False)
tsne_2d.to_csv(os.path.join(ART, "tsne_2d.csv"), index=False)
predictions.to_csv(os.path.join(ART, "predictions.csv"), index=False)
feature_importance.to_csv(os.path.join(ART, "feature_importance.csv"), index=False)

df_reset = df.reset_index().rename(columns={"index": "diamond_id"})
diamonds_search = df_reset[[
    "diamond_id", "carat", "cut", "color", "clarity",
    "price", "price_baseline_pred", "price_residual_pct", "Is_Overpriced", "Is_Undervalued",
]].copy()
diamonds_search.to_csv(os.path.join(ART, "diamonds_search.csv"), index=False)

X_test_sample = X_test.head(300).reset_index(drop=True)
X_test_sample.to_csv(os.path.join(ART, "X_test_sample.csv"), index=False)
ids_test_sample = df_reset.loc[idx_test, "diamond_id"].head(300).reset_index(drop=True)
ids_test_sample.to_frame(name="diamond_id").to_csv(os.path.join(ART, "ids_test_sample.csv"), index=False)

joblib.dump(model, os.path.join(ART, "model.pkl"))
joblib.dump(feature_cols_model, os.path.join(ART, "feature_cols_model.pkl"))

scale_params = {
    col: {"center": float(scaler.center_[i]), "scale": float(scaler.scale_[i])}
    for i, col in enumerate(numeric_cols)
}
with open(os.path.join(ART, "scale_params.json"), "w", encoding="utf-8") as f:
    json.dump(scale_params, f, ensure_ascii=False, indent=2)

with open(os.path.join(ART, "encoding_maps.json"), "w", encoding="utf-8") as f:
    json.dump(
        {"cut_order": cut_order, "color_order": color_order, "clarity_order": clarity_order},
        f, ensure_ascii=False, indent=2,
    )

calc_defaults = {
    "depth_median": float(df.loc[is_train, "depth"].median()),
    "table_median": float(df.loc[is_train, "table"].median()),
    "carat_min": float(df["carat"].min()),
    "carat_max": float(df["carat"].quantile(0.995)),
    "baseline_log_b0": float(_b0),
    "baseline_log_b1": float(_b1),
    "x_by_carat_slope": float(np.polyfit(df.loc[is_train, "carat"], df.loc[is_train, "x"], 1)[0]),
    "x_by_carat_intercept": float(np.polyfit(df.loc[is_train, "carat"], df.loc[is_train, "x"], 1)[1]),
    "y_by_carat_slope": float(np.polyfit(df.loc[is_train, "carat"], df.loc[is_train, "y"], 1)[0]),
    "y_by_carat_intercept": float(np.polyfit(df.loc[is_train, "carat"], df.loc[is_train, "y"], 1)[1]),
    "z_by_carat_slope": float(np.polyfit(df.loc[is_train, "carat"], df.loc[is_train, "z"], 1)[0]),
    "z_by_carat_intercept": float(np.polyfit(df.loc[is_train, "carat"], df.loc[is_train, "z"], 1)[1]),
}
with open(os.path.join(ART, "calc_defaults.json"), "w", encoding="utf-8") as f:
    json.dump(calc_defaults, f, ensure_ascii=False, indent=2)

## Krok 16 - Podsumowanie

In [18]:
print("Pipeline OK (bez wycieku danych w preprocessingu)")
print("n_rows:", len(df), "| duplicates removed:", n_duplicates)
print("zero-dim rows:", n_zero_dim, "| inconsistent measurement rows:", n_inconsistent)
print("R2:", round(r2, 4), "MAE:", round(mae, 4), "RMSE:", round(rmse, 4))
print("Baseline R2 (carat only):", round(baseline_r2, 4))
print("PCA 90% components:", n_components_90, "/", len(feature_cols_model))
print("Overpriced:", df["Is_Overpriced"].sum(), "| Undervalued:", df["Is_Undervalued"].sum())
print("Artifacts saved to:", os.path.abspath(ART))

Pipeline OK (bez wycieku danych w preprocessingu)
n_rows: 53794 | duplicates removed: 146
zero-dim rows: 19 | inconsistent measurement rows: 27
R2: 0.988 MAE: 0.0828 RMSE: 0.1095
Baseline R2 (carat only): 0.8283
PCA 90% components: 4 / 10
Overpriced: 1366 | Undervalued: 23
Artifacts saved to: /Users/mac/Desktop/Studia/Przetwarzanie danych i analiza wizualizacji/Projekt/artifacts


### Kluczowe wnioski

1. **Zero fabrykacji danych**: wszystkie wartości pochodzą z oryginalnego, publicznego zbioru diamonds.
2. **Realne problemy jakości danych**: 146 duplikatów, 19 zerowych wymiarów, 27 niespójności pomiarowych — wszystkie wykryte, nie stworzone.
3. **Poprawiony wyciek danych w preprocessingu**: podział train/test przesunięto przed imputację i skalowanie — obie te transformacje są teraz dopasowywane wyłącznie na zbiorze treningowym, zgodnie z zasadą z notatek teoretycznych.
4. **Wpływ poprawki na wynik**: R² na teście pozostaje ≈0.989 (praktycznie bez zmian) — przy tak dużym, jednorodnym zbiorze (53 794 wiersze) wyciek miał znikomy wpływ liczbowy, ale był realnym błędem metodologicznym, teraz wyeliminowanym.
5. **Model bazowy i detekcja anomalii cenowych** świadomie pozostają dopasowane do całego zbioru — to narzędzia opisowe (dashboard), a ich wyjścia są jawnie wykluczone z cech modelu ML, więc nie stanowią wycieku do ewaluacji.
6. **Dashboard**: kalkulator wyceny korzysta z parametrów skalowania i współczynników wyznaczonych wyłącznie na zbiorze treningowym.